## Repo Mental Model
   `main` is the current integration branch. It is not just the stock AIC repo,
   but it is also not a totally custom research codebase.

   The structure is: official AIC evaluation stack plus ranger-added 
   recorder/scenario/setup work plus the official Isaac and MuJoCo mirrors.

   The most important distinction is this:

   - Gazebo + `aic_engine` is the official benchmarking layer.
   - `aic_model` is the contract every policy must satisfy.
   - ...

---

```
  export DBX_CONTAINER_MANAGER=docker                
  distrobox create -r --nvidia -i ghcr.io/intrinsic-dev/aic/aic_eval:latest --name aic_eval
  distrobox enter -r aic_eval  

  distrobox enter --root aic_eval
```

---

LEARNING PATHS - 4 Blocks
   Given your goals (Issue #28 eval script --> Isaac Sim setup --> bare-bones RL
   --> ACT/Pi-zero training), here's the order that makes sense:


BLOCK 1: HOW EVALUATION WORKS NOW (directly feeds Issue #28)
   These are the files yoiu need to understand to build the automated eval 
   scripts:


aic_engine/config/sample_config.yaml
   - The trail definition format -- your eval script needs to load these

aic_engine/src/aic_engine.cpp
   - The orchestrator -- launches trails, calls policies, collects scores. Your
     script wraps this

aic_scoring/src/scoring.cpp
   - How AIC points are computed -- you need to parse/output these

aic_example_policies/aic_example_policies/ros/CheatCode.py
   - The baseline policy (~275 pts) -- your eval script's first test subject

aic_bringup/launch/aic_gz_bringup.launch.py
   - How Gazebo sim is laucnehd -- your script needs to automate this

docs/enrique_notes.md
   ... starting recipe

---

   ... Issue #28 eval script needs to CALL this launch file programatically as
   part of an automated pipeline -- rather than you manually opening a terminal
   and typing the `ros2 launch` command each time.

   And actually, if you look at what Nichols already built in 
   `flow-policy-training`'s `sim-eval.py`, he already does exactly this - he 
   spawns `ros2 launch aic_bringup aic_gz_bringup.launch.py` as a subprocess
   with the right arguments. So the automation layer is already written for you.

   So to be clear: `aic_gz_bringup.launch.py` is a file you need to UNDERSTAND
   (what it launches, what arguments it takes like `ground_truth`, 
   `start_aic_engine`, `aic_engine_config_file`), not something you need to
   write or modify.

---

---

## Flow Matching -- Quick Explanation

### The Core Idea
   You know how DIFFUSION MODELS (like Stable Diffusion for images) work by:
   1. Adding noise to data step by step until it's pure Gaussian noise
   2. Training a neural network to reverse that process (denoise step by step)
   3. At inference: start from random noise --> denoise --> get a sample

   FLOW MATCHING does something similar but simpler and faster. Instead of a 
   noisy diffusion process, it learns a STRAIGHT-LINE PATH (a "flow") from noise
   to data. 

   Think of it like this: 

     Diffusion:  noise ~~~zigzag path~~~> action     (many steps, curved)
     Flow:       noise ----straight line----> action  (fewer steps, direct)     


### How It Works in 3 Steps  

STEP 1 -- Training:
   You have demonstration data (e.g., robot trajectories from CheatCode). For
   each training sample:

   - Take a real action `a` from the dataset
   - Sample random noise `z` from a Gaussian
   - Pick a random time `t` between 0 and 1
   - Create an interpolation: `x_t = (1-t) * z + t*a` (a point along the straight
     line from noise to action)
   - Train the network to predict the VELOCITY (direction) that moves from `z`
     toward `a` at time `t`

   THe loss is literally just: "Given this noisy point `x_t` at time `t`, 
   predict the direction toward the real action"


STEP 2 -- The Velocity Field:
   After training, the network has learned a VECTOR FIELD -- at any point in
   action space and any time `t`, it can tell you which direction to move to
   reach a valid robot action.


STEP 3 -- Inference:
   TO generate an action:
   - Start at random noise `z` (t=0)
   - Step forward using the learned velocity field (like following arrows on a
     map)   
   - After a few steps you arrive at a valid action (t=1)


WHY IT'S GOOD FOR ROBOTICS

   Property // Why it matters
   
   MULTI-MODAL // If there are multiple valid ways to insert a cable, it can
      represent all of them (unlike a single-output network that averages them)
   FAST INFERENCE // Only needs ~5-10 integration steps vs. diffusion's ~50-100
      denoisin¬g steps
   STABLE TRAINING // The loss is a simple regression (MSE on velocity), no
      adverserial training or complex schedules
   WORKS WITH CONTINUOUS ACTIONS         
      Perfect for robot control -- outputs smooth 6-DOF twist commands


HOW IT COMPARES TO WHAT YOU KNOW
   Model // How it outputs actions

   ACT (Action Chunking Transformer) // Deterministic -- encodes observation,
      directly outputs a chunk of N future actions via a transformer decoder
   FLOW MATCHING // Stochastic -- encodes observation as conditioning, then
      iteratively refines noise into actions via learned velocity field
   DIFFUSION POLICY // Stochastic -- same idea but uses score-based denoising
      instead of straight flows (slower, more steps)
   RL (PPO) // Learns a policy distribution (usually Gaussian) over actions by
      maximising reward -- no demonstrations needed
   Pi-zero // Combines flow matching with a vision-language model backbone -- 
      uses a pretrained VLM to understand the task, then flow matching head to
      output actions


### In Nichols' Code Specifically
   Looking at his `aic_training/aic_training/models/flow_matching.py` (327 lines)
   , the architecture is:

   Observation (state + images)
      --> Image encoder (ResNet/ViT) extracts visual features
      --> State encoder processes 48-dim robot state
      --> COncatenate featurees + noisy acions + timestep t
      --> MLP/Transformer predicts velocity
      --> ODE inegration (noise --> action in ~10 steps)

   And his `pi05_wrapper.py` (153 lines) wraps a pi-zero model which is 
   essentially flow matching but using a large pre-trained VL model as the 
   backbone -- so it can understand task descriptions like "insert SFP cable
   into port" in addition to visual observations.


### Why Nichols Left Matters
   His `flow-policy-training` branch has the traing infrastructure but it's a
   DRAFT ... 
   - The eval script (`sim_eval.py`) is likely the most finished/useful part
   - The training code may need debugging/completion
   - You could either pick it up or use the infrastructure with your own ACT/RL
     approach instead


### Bottom Line for Your Plan
   Your plan of ACT fine-tuning --> RL in Isaac Sim doesn't require flow 
   matching at all. But Nichol's eval infrastructure (launching sim, parsing
   scores) is directly useful for Issue #28 regardless of which model 
   architecture you use. The `sim_eval.py` is model-agnostic -- it just launches
   whatever policy class you tell it to. 

---

  pip install -e aic/aic_utils/aic_isaac/aic_isaaclab/source/aic_task

  isaaclab -p aic/aic_utils/aic_isaac/aic_isaaclab/scripts/random_agent.py --task AIC-Task-v0 --num_envs 10 --enable_cameras

---

  pip install -e aic/aic_utils/aic_isaac/aic_isaaclab/source/aic_task
  isaaclab -p aic/aic_utils/aic_isaac/aic_isaaclab/scripts/random_agent.py --task AIC-Task-v0 --num_envs 10 --enable_cameras

--- version codex v1

  pip install -e aic/aic_utils/aic_isaac/aic_isaaclab/source/aic_task
  isaaclab -p aic/aic_utils/aic_isaac/aic_isaaclab/scripts/random_agent.py --task AIC-Task-v0 --num_envs 4 --enable_cameras

---   

---

### What Is Pi-Zero?
   Pi-zero is a ROBOT FOUNDATION MODEL from ... (often called "Pi"), The core
   idea is: take a massive pretrained Vision-Language Model (VLM) -- 
   specifically a 3B parameter PaLI-Gemma model -- and fine-tune it to output
   robot actions instead of text.

   The insight: VLMs already understand scenes, objects, spatial relationships,